# Full Behavioral Probing Replication (All Datasets)
This notebook downloads the repository and iterates through all 3 primary datasets (MovieLens, Beauty, Steam), training the required models and generating the probing results for each.

In [ ]:
import os

repo_dir = "/kaggle/working/repo"

if not os.path.exists(repo_dir):
    print("Cloning repository for the first time...")
    !git clone https://github.com/keshav31415/behaviour_probing.git {repo_dir}
else:
    print("Repository already exists. Pulling latest updates...")
    !cd {repo_dir} && git pull

%cd {repo_dir}

# 2. Mute the excessive \"loss in epoch...\" print statements in main.py so it doesn't flood the Kaggle output
!sed -i 's/print("loss in epoch/pass # print("loss in epoch/g' main.py

print("\nSetup complete! Data files ready to go:", os.listdir('data'))

In [ ]:
import glob
from IPython.display import Image, display

# --- CONFIGURATION ---
DRY_RUN = True
NUM_EPOCHS = 2 if DRY_RUN else 201

datasets = ["ml-1m", "Beauty", "Steam"]
models = ["SASRec", "GRU4Rec"]

print(f"Starting Automated Pipeline. DRY_RUN is {DRY_RUN} (Training for {NUM_EPOCHS} epochs).\n")

for dataset in datasets:
    print(f"\n{'='*80}")
    print(f"=== PROCESSING DATASET: {dataset.upper()} ===")
    print(f"{'='*80}\n")
    
    print("\n>>> [0/3] Regenerating Shuffled Dataset...")
    !python shuffle_seqs.py --dataset {dataset}
    
    for model in models:
        print(f"\n{'-'*40}")
        print(f"--- MODEL: {model} ---")
        print(f"{'-'*40}\n")
        
        print("\n>>> [1/3] Training Standard Model...")
        !python main.py --dataset={dataset} --model_type={model} --train_dir=default --maxlen=200 --dropout_rate=0.2 --device=cuda --num_epochs={NUM_EPOCHS}
        
        print("\n>>> [2/3] Training Shuffled Baseline...")
        !python main.py --dataset={dataset}_shuffled --model_type={model} --train_dir=shuffled --maxlen=200 --dropout_rate=0.2 --device=cuda --num_epochs={NUM_EPOCHS}
        
        print("\n>>> [3/3] Running Full Probing Analysis...")
        standard_models = glob.glob(f"{dataset}_default/{model}*.pth")
        shuffled_models = glob.glob(f"{dataset}_shuffled_shuffled/{model}*.pth")
        
        if not standard_models or not shuffled_models:
            print(f"Error: Could not find trained models for {dataset} - {model}. Skipping.")
            continue
            
        std_ckpt = standard_models[0]
        shuf_ckpt = shuffled_models[0]
        
        !python probing.py \
            --dataset {dataset} \
            --model_type {model} \
            --model_path {std_ckpt} \
            --shuffled_model_path {shuf_ckpt} \
            --run_mf \
            --run_coldstart \
            --run_behavior_analysis \
            --device cuda
            
        print(f"\n=== VISUAL RESULTS FOR {dataset.upper()} - {model} ===")
        images = glob.glob(f"probe_results/*{dataset}*.png")
        for img in images:
            display(Image(filename=img))
        print("\n\n")
